In [1]:
import pandas as pd
import numpy as np
from src.utils import read_query_file

# Import Data

In [7]:
df_debit_raw = pd.read_parquet("data/base/df_debit_all_clean_sample_v1.parquet")

In [10]:
df_debit_raw_2 = pd.read_parquet("data/base/df_debit_sample_v4.parquet")

In [11]:
df_debit_raw.shape, df_debit_raw_2.shape

((334635, 145), (380099, 145))

In [13]:
df_debit_raw_union = pd.concat([df_debit_raw, df_debit_raw_2], axis=0).reset_index(
    drop=True
)

C:\Users\Administrator\AppData\Local\Temp\3\ipykernel_7040\3134703544.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_debit_raw_union = pd.concat([df_debit_raw, df_debit_raw_2], axis=0).reset_index(drop=True)


In [14]:
df_debit_raw_union.shape

(714734, 145)

In [15]:
# Load MCC data

mcc_df = pd.read_csv("data/map/mcc_data.csv", sep=";", dtype={"Code": str})
mcc_df.rename({"Code": "MCC"}, axis=1, inplace=True)

# Drop Cols with High Missing Value Pct

In [16]:
from src.utils import get_features_by_missing_pct

In [17]:
selected_cols, summary_df = get_features_by_missing_pct(
    df_debit_raw_union,
    0.99,
    ["Debit_No", "Transaction Serial No", "Transaction Datetime", "Confirmed"],
)

In [5]:
selected_cols = [
    "Debit_No",
    "Transaction Serial No",
    "Transaction Datetime",
    "Transaction Amount",
    "MCC",
    "MCC Details",
    "MCC Trnx Category Code",
    "MCC Category",
    "Country Code",
    "Card Acceptor Terminal ID",
    "Card Acceptor Name",
    "Card Acceptor City",
    "Card Acceptor Region",
    "Card Acceptor Country Code",
    "Cat Card Acceptor Name",
    "Currency Code",
    "Confirmed",
    "CustomerAge",
    "AccountStatus",
    "CardStatus",
    "Balance",
    "AgeOfOpenAcctTxn",
    "AgeOfActiveCardTxn",
    "BDIStaff",
    "TotalTrxAmountL1D",
    "TotalTrxAmount15Mi",
    "HIghRiskCustomer",
    "IsBlacklistAccount",
    "AccessCode",
    "NumOfTrxL1D",
    "IsHighRiskCountry",
    "IsBlacklistMerchant",
    "IsWatchListMerchant",
    "CardProduct",
    "CustomerSex",
    "IsBlacklistCardNo",
    "Currency",
    "POSMode",
    "PINEntryCapability",
    "IsWhiteListMerchant",
    "IsBDIEmail",
    "IsWhiteListCardNo",
    "IsWhiteListAccountNo",
    "IsWatchListAccountNo",
    "TransactionType",
    "TotalTrxAmount10Mi",
    "TotalTrxAmountL5min",
    "TotalTrxAmountContactless",
    "IsVAAccount",
    "IsExcludeTeminalId",
    "isBDICard",
    "VCardDataInputCapability",
    "CustomerNationality",
    "TotalTrxAmountContactlessPerDay",
    "IsTopUp",
    "IsBillPayment",
    "IsPaymentToVA",
    "TotalTrxAmountTopUp1",
    "TotalTrxAmountTopUp2",
    "TotalTrxAmountVA1",
    "TotalTrxAmountVA2",
    "IsQRtrx",
    "IsDCashTrx",
    "IsFTTrx",
    "TotalTrxAmountTrf",
    "CountTrxTrf",
    "TotalTrxAmountTrf2",
    "CountTrxTrf2",
    "NumOfTrxL15Min",
    "TimeDiffCurrPrev",
    "CountTrxEOD",
]

In [18]:
pd.set_option("display.max_rows", None)
summary_df[
    ~summary_df.feature.isin(
        ["Debit_No", "Transaction Serial No", "Transaction Datetime", "Confirmed"]
        + selected_cols
    )
].reset_index(drop=True)

,feature,missing_pct
0,CustomerAvgIncome,1.000000
1,AgeOfOpenAcctActiveCard,1.000000
2,TrfToBDIStaff,1.000000
3,AgeOfRegDateTxn,1.000000
4,IsSDBPastDue,1.000000
5,FlagOutBranch,1.000000
6,isTDHoldAmount,1.000000
7,isProgramHoldAmount,1.000000
8,IsOfficeHour,1.000000
9,VALAS,1.000000


In [20]:
df_debit_raw_union = df_debit_raw_union[
    ["Debit_No", "Transaction Serial No", "Transaction Datetime", "Confirmed"]
    + selected_cols
].copy()

In [21]:
df_debit_raw_union.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714734 entries, 0 to 714733
Data columns (total 69 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Debit_No                         714734 non-null  object        
 1   Transaction Serial No            714734 non-null  int64         
 2   Transaction Datetime             714734 non-null  datetime64[ns]
 3   Confirmed                        2770 non-null    float64       
 4   Transaction Amount               684274 non-null  float64       
 5   MCC                              188098 non-null  object        
 6   Country Code                     188098 non-null  object        
 7   Card Acceptor Terminal ID        188098 non-null  object        
 8   Card Acceptor Name               188098 non-null  object        
 9   Card Acceptor City               188098 non-null  object        
 10  Card Acceptor Region             188098 non-

# Feature Engineering Logics

In [7]:
from src.calculation_features import (
    generate_rolling_features,
    calculate_time_differences,
)

## Value Mapping Steps

1. MCC value mapping: 
- MCC to MCC Details
- MCC to MCC Transaction Category Code
- MCC to MCC Category
2. Value mapping Card Acceptor Name to bigger group (ecommerce, online ads, etc.)

In [8]:
from src.utils import clean_categorize_merchant_name
from tqdm import tqdm

In [9]:
# 1. Value mapping MCC_code to MCC Details, Trnx Category Code, Category

df_debit_clean = df_debit_raw.merge(
    mcc_df[["MCC", "Description", "Transaction Category Code", "MCC Category"]],
    on="MCC",
    how="left",
)
df_debit_clean.rename(
    columns={
        "Description": "MCC Details",
        "Transaction Category Code": "MCC Trnx Category Code",
    },
    inplace=True,
)

In [10]:
# 2. Value mapping Card Acceptor Name/Card Acceptor Name to bigger group (ecommerce, online ads, etc.)
tqdm.pandas(desc="Clean and Categorize Acceptor Name Progress")
df_debit_clean["Cat Card Acceptor Name"] = df_debit_clean[
    "Card Acceptor Name"
].progress_apply(clean_categorize_merchant_name)

Clean and Categorize Acceptor Name Progress: 100%|███████████████████████████| 334635/334635 [00:46<00:00, 7180.37it/s]


In [14]:
channel_cols = [
    "Debit_No",
    "Transaction Serial No",
    "Transaction Datetime",
    "Transaction Amount",
    "MCC",
    "MCC Details",
    "MCC Trnx Category Code",
    "MCC Category",
    "Country Code",
    "Card Acceptor City",
    "Card Acceptor Region",
    "Card Acceptor Country Code",
    "Cat Card Acceptor Name",
    "Currency Code",
    "Confirmed",
]

In [12]:
# df_debit_clean.to_parquet("data/base/df_debit_clean_v2.parquet")
df_debit_clean[selected_cols].to_parquet(
    "data/base/df_debit_clean_all_sample_v1.parquet"
)

In [ ]:
# df_debit_clean = pd.read_parquet("data/base/df_debit_clean.parquet")

In [15]:
df_debit_filter = df_debit_clean[channel_cols].copy()

# Downcasting Float Features

In [16]:
import gc


def preprocess_data(df):
    print("Starting preprocessing...")
    df_processed = df.copy()

    # identify and safely downcast float columns
    float_cols = df_processed.select_dtypes(include=["float64", "float32"]).columns
    print(f"Float columns to downcast: {len(float_cols)}")
    for col in float_cols:
        try:
            df_processed[col] = pd.to_numeric(df_processed[col], downcast="float")
        except Exception as e:
            print(f"Warning: failed to downcast {col}: {e}")

    for col in df_processed.columns:
        if isinstance(col, str):
            try:
                if df_processed[col].dtype == "object":
                    # treat pure whitespaces as missing
                    df_processed[col] = df_processed[col].replace(
                        r"^\s*$", np.nan, regex=True
                    )

                    unique_vals = df_processed[col].dropna().unique()
                    if set(unique_vals).issubset({"Y", "N"}):
                        print(f"Mapping Y/N to 1/0 in column: {col}")
                        df_processed[col] = df_processed[col].map({"N": 0, "Y": 1})
                if col.startswith("Is") and df_processed[col].dtype in [
                    "object",
                    "int64",
                ]:
                    print(f"Mapping Is* column Y/N to 1/0: {col}")
                    df_processed[col] = df_processed[col].map({"N": 0, "Y": 1})
            except Exception as e:
                print(f"Warning processing column {col}: {e}")

    print(
        "Estimated memory usage (MB):", df_processed.memory_usage(deep=True).sum() / 1e6
    )
    gc.collect()
    return df_processed

In [17]:
df_debit_filter = preprocess_data(df_debit_filter)

Starting preprocessing...
Float columns to downcast: 1
Mapping Y/N to 1/0 in column: Confirmed
Estimated memory usage (MB): 174.837986


## Time Difference Features

In [19]:
from src.debit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
    monetary_config_4,
    monetary_config_5,
    monetary_config_6,
    unique_count_config,
)
import warnings

warnings.filterwarnings("ignore")

all_monetary_configs = (
    monetary_config_1
    + monetary_config_2
    + monetary_config_3
    + monetary_config_4
    + monetary_config_5
    + monetary_config_6
)

In [20]:
df_time_diff = calculate_time_differences(
    df=df_debit_filter,
    datetime_col="Transaction Datetime",
    groupby_col="Debit_No",
    time_window=time_windows,
    config=time_shift_config,
)

Calculating rolling averages: 100%|██████████████████████████████████████████████████████| 7/7 [00:11<00:00,  1.61s/it]


In [21]:
df_time_diff.to_parquet("data/feature_engineering/debit/v2/clean/df_time_diff.parquet")

## Frequency Features

In [22]:
df_freq = generate_rolling_features(
    df_debit_filter,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress: 100%|███████████████████████████████████████████████████████████| 7/7 [01:10<00:00, 10.11s/it]


In [23]:
df_freq.to_parquet("data/feature_engineering/debit/v2/clean/df_freq.parquet")

## Monetary

In [24]:
df_monetary = generate_rolling_features(
    df_debit_filter,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=all_monetary_configs,
)

Feature Config Progress: 100%|█████████████████████████████████████████████████████████| 18/18 [03:01<00:00, 10.06s/it]


In [25]:
df_monetary.to_parquet("data/feature_engineering/debit/v2/clean/df_monetary.parquet")

## Unique Count

In [26]:
df_debit_filter["MCC Num"], uniques = df_debit_filter["MCC"].factorize()
df_debit_filter["Debit_No Num"], uniques = df_debit_filter["Debit_No"].factorize()

In [27]:
df_unique_count = generate_rolling_features(
    df_debit_filter,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=unique_count_config,
)

Feature Config Progress: 100%|██████████████████████████████████████████████████████████| 4/4 [43:08<00:00, 647.09s/it]


In [28]:
df_unique_count.to_parquet(
    "data/feature_engineering/debit/v2/clean/df_unique_count.parquet"
)